In [4]:
import os

In [6]:
print("Current directory:", os.getcwd())
print("Main folders:")
print(os.listdir("."))

Current directory: C:\Users\GOD\PycharmProjects\dental_p\MOPG-7
Main folders:
['annotations', 'data preprocessing.ipynb', 'explore.ipynb', 'images', 'labels_yolo', 'metadata']


In [8]:
print("Images folders:")
print(os.listdir("images")[:10])

print("\nYOLO labels folders:")
print(os.listdir("labels_yolo")[:10])

print("\nAnnotations folders:")
print(os.listdir("annotations")[:10])

print("\nMetadata folders:")
print(os.listdir("metadata")[:10])

Images folders:
['test', 'train', 'val']

YOLO labels folders:
['test', 'train', 'val']

Annotations folders:
['instances_test.json', 'instances_train.json', 'instances_val.json']

Metadata folders:
['class_definitions.csv', 'image_metadata.csv', 'split_assignments.csv']


In [10]:
for split in ["train", "val", "test"]:

    image_path = os.path.join("images", split)
    label_path = os.path.join("labels_yolo", split)

    images = os.listdir(image_path)
    labels = os.listdir(label_path)

    print(f"\n{split.upper()}")
    print("-" * 30)

    print("Number of images:", len(images))
    print("Number of labels:", len(labels))

    print("Sample images:", images[:5])
    print("Sample labels:", labels[:5])


TRAIN
------------------------------
Number of images: 1676
Number of labels: 1676
Sample images: ['00001.jpg', '000010.jpg', '0000100.jpg', '00001001.jpg', '00001002.jpg']
Sample labels: ['00001.txt', '000010.txt', '0000100.txt', '00001001.txt', '00001002.txt']

VAL
------------------------------
Number of images: 209
Number of labels: 209
Sample images: ['00001015.jpg', '00001019.jpg', '00001041.jpg', '0000105.jpg', '00001064.jpg']
Sample labels: ['00001015.txt', '00001019.txt', '00001041.txt', '0000105.txt', '00001064.txt']

TEST
------------------------------
Number of images: 210
Number of labels: 210
Sample images: ['00001000.jpg', '00001021.jpg', '00001032.jpg', '00001037.jpg', '00001053.jpg']
Sample labels: ['00001000.txt', '00001021.txt', '00001032.txt', '00001037.txt', '00001053.txt']


In [12]:
from pathlib import Path

for split in ["train", "val", "test"]:

    image_dir = Path("images") / split
    label_dir = Path("labels_yolo") / split

    image_stems = {p.stem for p in image_dir.iterdir()
                   if p.suffix.lower() in [".jpg", ".jpeg", ".png"]}

    label_stems = {p.stem for p in label_dir.glob("*.txt")}

    missing_labels = image_stems - label_stems
    extra_labels = label_stems - image_stems

    empty_labels = 0
    invalid_labels = 0
    total_boxes = 0

    for file in label_dir.glob("*.txt"):

        with open(file, "r") as f:
            lines = f.readlines()

        if len(lines) == 0:
            empty_labels += 1

        for line in lines:
            values = line.strip().split()

            if len(values) != 5:
                invalid_labels += 1
                continue

            try:
                class_id = int(values[0])
                x, y, w, h = map(float, values[1:])

                if (class_id < 0 or class_id > 6 or
                    not all(0 <= v <= 1 for v in [x, y, w, h]) or
                    w <= 0 or h <= 0):
                    invalid_labels += 1
                else:
                    total_boxes += 1

            except ValueError:
                invalid_labels += 1

    print(f"\n{split.upper()}")
    print("-" * 35)
    print("Missing labels:", len(missing_labels))
    print("Extra labels:", len(extra_labels))
    print("Empty label files:", empty_labels)
    print("Invalid annotations:", invalid_labels)
    print("Valid bounding boxes:", total_boxes)


TRAIN
-----------------------------------
Missing labels: 0
Extra labels: 1
Empty label files: 18
Invalid annotations: 0
Valid bounding boxes: 7823

VAL
-----------------------------------
Missing labels: 0
Extra labels: 0
Empty label files: 2
Invalid annotations: 0
Valid bounding boxes: 923

TEST
-----------------------------------
Missing labels: 0
Extra labels: 0
Empty label files: 0
Invalid annotations: 0
Valid bounding boxes: 1088


In [20]:
from pathlib import Path

for split in ["train", "val", "test"]:

    image_dir = Path("images") / split
    label_dir = Path("labels_yolo") / split

    image_extensions = [".jpg", ".jpeg", ".png", ".bmp"]

    image_stems = {
        p.stem for p in image_dir.iterdir()
        if p.suffix.lower() in image_extensions
    }

    label_stems = {p.stem for p in label_dir.glob("*.txt")}

    print(f"\n{split.upper()}")
    print("Number of images:", len(image_stems))
    print("Number of labels:", len(label_stems))
    print("Missing labels:", len(image_stems - label_stems))
    print("Extra labels:", len(label_stems - image_stems))


TRAIN
Number of images: 1676
Number of labels: 1676
Missing labels: 0
Extra labels: 0

VAL
Number of images: 209
Number of labels: 209
Missing labels: 0
Extra labels: 0

TEST
Number of images: 210
Number of labels: 210
Missing labels: 0
Extra labels: 0


In [16]:
import pandas as pd
classes = pd.read_csv("metadata/class_definitions.csv")
display(classes)

,class_id,class_name,annotation_count,description
0,0,Missing Teeth,2609,Congenitally missing or previously extracted p...
1,1,Dental Crown,1984,Prosthetic dental crowns (full-coverage restor...
2,2,Root Canal,1956,Endodontically treated teeth exhibiting radiop...
3,3,Caries,1410,Carious lesions presenting as radiolucent area...
4,4,Broken Down Teeth,795,Teeth with extensive structural loss (large ca...
5,5,Wisdom Teeth,869,"Third molars (wisdom teeth), regardless of eru..."
6,6,Healthy OPG,211,Panoramic radiographs showing normal dental st...


In [18]:
pipimport json
from pathlib import Path

# Load training annotations
with open("annotations/instances_train.json", encoding="utf-8") as f:
    train_data = json.load(f)

# Map image filenames to image IDs
image_map = {
    Path(img["file_name"]).stem: img["id"]
    for img in train_data["images"]
}

# Count annotations for each image
annotation_counts = {}

for ann in train_data["annotations"]:
    img_id = ann["image_id"]
    annotation_counts[img_id] = annotation_counts.get(img_id, 0) + 1

# Check extra label
extra_label = "00001497"
print("Extra label:", extra_label)
print("Exists in JSON images:", extra_label in image_map)

if extra_label in image_map:
    img_id = image_map[extra_label]
    print("Image ID:", img_id)
    print("Annotations in JSON:", annotation_counts.get(img_id, 0))

# Check empty training labels
empty_labels = [
    p.stem for p in Path("labels_yolo/train").glob("*.txt")
    if p.stat().st_size == 0
]

print("\nEmpty training labels:")
for stem in empty_labels:
    if stem in image_map:
        img_id = image_map[stem]
        print(stem, "-> JSON annotations:", annotation_counts.get(img_id, 0))
    else:
        print(stem, "-> Image not found in JSON")

Extra label: 00001497
Exists in JSON images: True
Image ID: 652
Annotations in JSON: 5

Empty training labels:
00001120 -> JSON annotations: 0
00001209 -> JSON annotations: 0
00001373 -> JSON annotations: 0
00001480 -> JSON annotations: 0
0000179 -> JSON annotations: 0
0000264 -> JSON annotations: 0
0000445 -> JSON annotations: 0
0000682 -> JSON annotations: 0
0000737 -> JSON annotations: 0
0000818 -> JSON annotations: 0
ibn231 -> JSON annotations: 0
ibn306 -> JSON annotations: 0
ibn322 -> JSON annotations: 0
ibn347 -> JSON annotations: 0
ibn378 -> JSON annotations: 0
ibn395 -> JSON annotations: 0
ibn479 -> JSON annotations: 0
ibn524 -> JSON annotations: 0


In [21]:
# Find image information in JSON
img_info = next(
    img for img in train_data["images"]
    if Path(img["file_name"]).stem == "00001497"
)

print("Image information:")
print(img_info)

# Check whether image exists
image_file = Path("images/train") / Path(img_info["file_name"]).name

print("\nExpected image path:", image_file)
print("Image exists:", image_file.exists())

# Read YOLO label
label_file = Path("labels_yolo/train/00001497.txt")

print("\nYOLO label content:")
with open(label_file, "r") as f:
    print(f.read())

# Show JSON annotations
img_id = img_info["id"]

print("\nJSON annotations:")
for ann in train_data["annotations"]:
    if ann["image_id"] == img_id:
        print(
            "Class ID:", ann["category_id"],
            "Bounding box:", ann["bbox"]
        )

Image information:
{'id': 652, 'file_name': 'train/00001497.bmp', 'width': 2864, 'height': 1504}

Expected image path: images\train\00001497.bmp
Image exists: True

YOLO label content:
5 0.290154 0.429189 0.044693 0.092420
5 0.269029 0.655585 0.056913 0.126330
5 0.751920 0.441822 0.040852 0.087101
5 0.773568 0.670878 0.062500 0.083777
2 0.722765 0.687168 0.029330 0.095080


JSON annotations:
Class ID: 5 Bounding box: [767.001, 576.0, 128.001, 139.0]
Class ID: 5 Bounding box: [689.0, 891.0, 162.999, 190.0]
Class ID: 5 Bounding box: [2094.999, 599.0, 117.0, 131.0]
Class ID: 5 Bounding box: [2125.999, 946.0, 179.0, 126.001]
Class ID: 2 Bounding box: [2027.998, 962.001, 84.001, 143.0]


In [23]:
import shutil
from pathlib import Path

if not Path("labels").exists():
    shutil.copytree("labels_yolo", "labels")
    print("Labels folder created successfully!")
else:
    print("Labels folder already exists. No changes made.")

Labels folder already exists. No changes made.


In [24]:
for split in ["train", "val", "test"]:
    print(
        split,
        "images:", len(list((Path("images") / split).iterdir())),
        "labels:", len(list((Path("labels") / split).glob("*.txt")))
    )

train images: 1676 labels: 1676
val images: 209 labels: 209
test images: 210 labels: 210


In [1]:
import ultralytics

print("Ultralytics version:", ultralytics.version)

Creating new Ultralytics Settings v0.0.8 file  
View Ultralytics Settings with 'yolo settings' or at 'C:\Users\mahdi\AppData\Roaming\Ultralytics\settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings


AttributeError: module ultralytics has no attribute version

In [2]:
import ultralytics

print("Ultralytics version:", ultralytics.version)

AttributeError: module ultralytics has no attribute version

In [3]:
from ultralytics import YOLO

model = YOLO("yolo11n.pt")

model.train(
    data="data.yaml",
    epochs=100,
    imgsz=640,
    batch=2,
    device="cpu"
)

Ultralytics 8.4.203  Python-3.13.17 torch-2.15.0+cpu CPU (13th Gen Intel Core i5-13450HX)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=2, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=data.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64, nms=None, opset=None, optimize=False, optimizer=auto, ov

RuntimeError: Dataset 'data.yaml' error  'data.yaml' does not exist

In [ ]:

import torch

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():

    print("GPU:", torch.cuda.get_device_name(0))

    print(
        "VRAM:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3, 2
        ),
        "GB"
    )

    x = torch.rand(3, 3, device="cuda")
    print("GPU calculation:", x @ x)

else:
    print("GPU NOT AVAILABLE")
